In [0]:
# Identificação do notebook de análise final do MVP.
# Este notebook consulta as tabelas existentes e não altera os dados do pipeline.

from datetime import datetime

data_execucao = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Análise final do MVP")
print(f"Data e hora da execução: {data_execucao}")
print("Ambiente: Databricks")
print("Objetivo: validar tabelas, qualidade dos dados e resultados do pipeline.")

Análise final do MVP
Data e hora da execução: 2026-09-20 13:46:46
Ambiente: Databricks
Objetivo: validar tabelas, qualidade dos dados e resultados do pipeline.


In [0]:
# Relação das tabelas esperadas no MVP.
# A validação será feita individualmente para identificar tabelas existentes ou ausentes.

tabelas_esperadas = [
    "workspace.default.dim_atleta",
    "workspace.default.dim_atleta_nome_restrito",
    "workspace.default.dim_atleta_trainingpeaks",
    "workspace.default.dim_condicao_clinica",
    "workspace.default.dim_desempenho_corrida",
    "workspace.default.dim_habitos_suplementacao",
    "workspace.default.dim_hidratacao_nutricao",
    "workspace.default.fato_sessao_treino",
    "workspace.default.fato_sessao_treino_zonas",
    "workspace.default.fato_alerta_clinico",
    "workspace.default.dim_regra_suplementacao_zona",
    "workspace.default.dim_protocolo_suplementacao_base",
    "workspace.default.gold_prescricao_diaria",
    "workspace.default.fato_documento_prescricao"
]

print(f"Quantidade de tabelas esperadas: {len(tabelas_esperadas)}")

Quantidade de tabelas esperadas: 14


In [0]:
# Validação da existência das tabelas no catálogo do Databricks.
# Esta célula não cria nem modifica nenhuma tabela.

resultado_tabelas = []

for nome_tabela in tabelas_esperadas:
    try:
        existe = spark.catalog.tableExists(nome_tabela)
    except Exception as erro:
        existe = False
        print(f"Não foi possível consultar {nome_tabela}: {erro}")

    resultado_tabelas.append(
        {
            "tabela": nome_tabela,
            "existe": existe
        }
    )

df_validacao_tabelas = spark.createDataFrame(resultado_tabelas)

display(
    df_validacao_tabelas.orderBy("tabela")
)

existe,tabela
true,workspace.default.dim_atleta
true,workspace.default.dim_atleta_nome_restrito
true,workspace.default.dim_atleta_trainingpeaks
true,workspace.default.dim_condicao_clinica
true,workspace.default.dim_desempenho_corrida
true,workspace.default.dim_habitos_suplementacao
true,workspace.default.dim_hidratacao_nutricao
true,workspace.default.dim_protocolo_suplementacao_base
true,workspace.default.dim_regra_suplementacao_zona
true,workspace.default.fato_alerta_clinico


In [0]:
# Resumo das tabelas existentes e ausentes.
# A contagem de registros é feita somente para as tabelas encontradas.

resumo_tabelas = []

for item in resultado_tabelas:
    nome_tabela = item["tabela"]
    existe = item["existe"]

    if existe:
        try:
            quantidade_registros = spark.table(nome_tabela).count()
        except Exception as erro:
            quantidade_registros = None
            print(f"Erro ao contar registros de {nome_tabela}: {erro}")
    else:
        quantidade_registros = None

    resumo_tabelas.append(
        {
            "tabela": nome_tabela,
            "existe": existe,
            "quantidade_registros": quantidade_registros
        }
    )

df_resumo_tabelas = spark.createDataFrame(resumo_tabelas)

display(
    df_resumo_tabelas.orderBy("tabela")
)

existe,quantidade_registros,tabela
true,3,workspace.default.dim_atleta
true,3,workspace.default.dim_atleta_nome_restrito
true,1,workspace.default.dim_atleta_trainingpeaks
true,3,workspace.default.dim_condicao_clinica
true,3,workspace.default.dim_desempenho_corrida
true,3,workspace.default.dim_habitos_suplementacao
true,3,workspace.default.dim_hidratacao_nutricao
true,8,workspace.default.dim_protocolo_suplementacao_base
true,7,workspace.default.dim_regra_suplementacao_zona
true,8,workspace.default.fato_alerta_clinico


In [0]:
# Funções auxiliares para consultar tabelas com segurança.
# A função evita que a análise seja interrompida caso uma coluna não exista.

def tabela_existe(nome_tabela):
    # Verifica se uma tabela está disponível no catálogo.
    return spark.catalog.tableExists(nome_tabela)


def colunas_tabela(nome_tabela):
    # Retorna os nomes das colunas de uma tabela existente.
    if not tabela_existe(nome_tabela):
        return []
    return spark.table(nome_tabela).columns


def possui_colunas(nome_tabela, colunas_necessarias):
    # Confirma se todas as colunas solicitadas existem na tabela.
    colunas_disponiveis = set(colunas_tabela(nome_tabela))
    return set(colunas_necessarias).issubset(colunas_disponiveis)


print("Funções auxiliares criadas com sucesso.")

Funções auxiliares criadas com sucesso.


In [0]:
# Exibição do esquema das principais tabelas do MVP.
# Esta evidência pode ser utilizada para apoiar o catálogo de dados.

tabelas_principais = [
    "workspace.default.dim_atleta",
    "workspace.default.fato_sessao_treino",
    "workspace.default.fato_alerta_clinico",
    "workspace.default.gold_prescricao_diaria",
    "workspace.default.fato_documento_prescricao"
]

for nome_tabela in tabelas_principais:
    print(f"\n===== {nome_tabela} =====")

    if tabela_existe(nome_tabela):
        spark.table(nome_tabela).printSchema()
    else:
        print("Tabela não encontrada.")


===== workspace.default.dim_atleta =====
root
 |-- id_atleta: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- altura_cm: double (nullable = true)
 |-- peso_kg: double (nullable = true)
 |-- objetivo_principal: string (nullable = true)
 |-- prova_alvo: string (nullable = true)
 |-- frequencia_treino_semanal: string (nullable = true)
 |-- horario_treino: string (nullable = true)
 |-- trainingpeaks_link: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- idade_anos: long (nullable = true)


===== workspace.default.fato_sessao_treino =====
root
 |-- data_treino: string (nullable = true)
 |-- titulo_treino: string (nullable = true)
 |-- id_treino: string (nullable = true)
 |-- modalidade: string (nullable = true)
 |-- descricao_treino: string (nullable = true)
 |-- duracao_minutos: double (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- fc_media_bpm: integer (nullable = true)
 |-- 

In [0]:
# Cálculo dos principais indicadores gerais do MVP.
# Os indicadores são calculados somente quando as respectivas tabelas existem.

def contar_tabela(nome_tabela):
    # Retorna a quantidade de registros ou None quando a tabela não existe.
    if not tabela_existe(nome_tabela):
        return None
    return spark.table(nome_tabela).count()


indicadores_gerais = [
    {
        "indicador": "Atletas na dimensão principal",
        "valor": contar_tabela("workspace.default.dim_atleta")
    },
    {
        "indicador": "Sessões de treinamento",
        "valor": contar_tabela("workspace.default.fato_sessao_treino")
    },
    {
        "indicador": "Sessões com zonas",
        "valor": contar_tabela("workspace.default.fato_sessao_treino_zonas")
    },
    {
        "indicador": "Alertas clínicos",
        "valor": contar_tabela("workspace.default.fato_alerta_clinico")
    },
    {
        "indicador": "Prescrições diárias",
        "valor": contar_tabela("workspace.default.gold_prescricao_diaria")
    },
    {
        "indicador": "Documentos de prescrição",
        "valor": contar_tabela("workspace.default.fato_documento_prescricao")
    }
]

df_indicadores_gerais = spark.createDataFrame(indicadores_gerais)

display(df_indicadores_gerais)

indicador,valor
Atletas na dimensão principal,3
Sessões de treinamento,4
Sessões com zonas,2
Alertas clínicos,8
Prescrições diárias,4
Documentos de prescrição,null


In [0]:
# Validação do vínculo entre atletas e sessões de treinamento.
# O identificador analítico utilizado é id_atleta.

nome_dim_atleta = "workspace.default.dim_atleta"
nome_fato_treino = "workspace.default.fato_sessao_treino"

if tabela_existe(nome_dim_atleta) and tabela_existe(nome_fato_treino):
    df_atletas = spark.table(nome_dim_atleta)
    df_treinos = spark.table(nome_fato_treino)

    if possui_colunas(nome_dim_atleta, ["id_atleta"]) and possui_colunas(
        nome_fato_treino, ["id_atleta"]
    ):
        total_atletas = df_atletas.select("id_atleta").distinct().count()
        atletas_com_treino = df_treinos.select("id_atleta").distinct().count()

        print(f"Atletas cadastrados: {total_atletas}")
        print(f"Atletas com sessões de treino: {atletas_com_treino}")

        display(
            df_treinos.groupBy("id_atleta")
            .count()
            .orderBy("id_atleta")
        )
    else:
        print("A coluna id_atleta não está disponível em uma das tabelas.")
else:
    print("Uma ou ambas as tabelas principais não foram encontradas.")

Atletas cadastrados: 3
Atletas com sessões de treino: 1


id_atleta,count
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,4


In [0]:
# Identificação de sessões sem id_atleta.
# Uma sessão sem identificador não pode ser corretamente associada ao atleta.

if tabela_existe(nome_fato_treino):
    df_treinos = spark.table(nome_fato_treino)

    if possui_colunas(nome_fato_treino, ["id_atleta"]):
        df_sessoes_sem_atleta = df_treinos.filter(
            "id_atleta IS NULL OR TRIM(CAST(id_atleta AS STRING)) = ''"
        )

        quantidade_sessoes_sem_atleta = df_sessoes_sem_atleta.count()

        print(
            f"Sessões sem id_atleta: {quantidade_sessoes_sem_atleta}"
        )

        display(df_sessoes_sem_atleta)
    else:
        print("A coluna id_atleta não existe na tabela de sessões.")

Sessões sem id_atleta: 0


data_treino,titulo_treino,id_treino,modalidade,descricao_treino,duracao_minutos,distancia_km,fc_media_bpm,fc_max_bpm,tss,id_regra_nutricao,zona_fc_predominante,classificacao_intensidade,minutos_zona_baixa,minutos_zona_moderada,minutos_zona_alta,id_atleta


In [0]:
# Verificação de duplicidade dos treinamentos.
# O identificador esperado da sessão é id_treino.

if tabela_existe(nome_fato_treino):
    df_treinos = spark.table(nome_fato_treino)

    if possui_colunas(nome_fato_treino, ["id_treino"]):
        total_registros = df_treinos.count()
        total_ids_distintos = df_treinos.select("id_treino").distinct().count()
        total_duplicados = total_registros - total_ids_distintos

        print(f"Total de registros: {total_registros}")
        print(f"Total de id_treino distintos: {total_ids_distintos}")
        print(f"Possíveis registros duplicados: {total_duplicados}")

        display(
            df_treinos.groupBy("id_treino")
            .count()
            .filter("count > 1")
            .orderBy("count", ascending=False)
        )
    else:
        print("A coluna id_treino não existe na tabela de sessões.")

Total de registros: 4
Total de id_treino distintos: 4
Possíveis registros duplicados: 0


id_treino,count


In [0]:
# Distribuição das sessões por modalidade.
# A análise utiliza a coluna modalidade confirmada na fato_sessao_treino.

if tabela_existe(nome_fato_treino):
    df_treinos = spark.table(nome_fato_treino)

    if possui_colunas(nome_fato_treino, ["modalidade"]):
        display(
            df_treinos.groupBy("modalidade")
            .count()
            .orderBy("count", ascending=False)
        )
    else:
        print("A coluna modalidade não existe na tabela de sessões.")

modalidade,count
Run,4


In [0]:
# Resumo das principais métricas de treinamento.
# As métricas são analisadas somente quando estão disponíveis.

metricas_treino = [
    coluna
    for coluna in ["duracao_minutos", "distancia_km", "fc_media_bpm", "fc_max_bpm", "tss"]
    if coluna in colunas_tabela(nome_fato_treino)
]

if tabela_existe(nome_fato_treino) and len(metricas_treino) > 0:
    df_treinos = spark.table(nome_fato_treino)

    display(
        df_treinos.select(metricas_treino).summary()
    )
else:
    print("Não foram encontradas métricas numéricas para análise.")

summary,duracao_minutos,distancia_km,fc_media_bpm,fc_max_bpm,tss
count,4,4,0,0,0
mean,60.0,13.39,null,null,null
stddev,34.64101615137755,1.628127759114745,null,null,null
min,30.0,11.98,null,null,null
25%,30.0,11.98,null,null,null
50%,30.0,11.98,null,null,null
75%,90.0,14.8,null,null,null
max,90.0,14.8,null,null,null


In [0]:
# Análise dos alertas clínicos gerados pelo pipeline.
# As colunas abaixo foram confirmadas na fato_alerta_clinico.

nome_fato_alerta = "workspace.default.fato_alerta_clinico"

if tabela_existe(nome_fato_alerta):
    df_alertas = spark.table(nome_fato_alerta)

    if possui_colunas(nome_fato_alerta, ["tipo_alerta"]):
        print("Distribuição por tipo de alerta:")
        display(
            df_alertas.groupBy("tipo_alerta")
            .count()
            .orderBy("count", ascending=False)
        )

    if possui_colunas(nome_fato_alerta, ["nivel_prioridade"]):
        print("Distribuição por nível de prioridade:")
        display(
            df_alertas.groupBy("nivel_prioridade")
            .count()
            .orderBy("count", ascending=False)
        )

    if possui_colunas(nome_fato_alerta, ["status_revisao"]):
        print("Distribuição por status de revisão:")
        display(
            df_alertas.groupBy("status_revisao")
            .count()
            .orderBy("count", ascending=False)
        )
else:
    print("A tabela de alertas clínicos não foi encontrada.")

Distribuição por tipo de alerta:


tipo_alerta,count
SEGURANCA_CARDIOVASCULAR,3
FARMACO,3
ALERGIA_INTOLERANCIA,2


Distribuição por nível de prioridade:


nivel_prioridade,count
CRITICO,5
ALTO,3


Distribuição por status de revisão:


status_revisao,count
PENDENTE_REVISAO_FARMACEUTICA,8


In [0]:
# Validação da tabela Gold de prescrições.
# A estrutura das colunas é exibida para apoiar a análise e o catálogo de dados.

nome_gold = "workspace.default.gold_prescricao_diaria"

if tabela_existe(nome_gold):
    df_gold = spark.table(nome_gold)

    print(f"Registros na Gold: {df_gold.count()}")
    print("Colunas disponíveis na Gold:")
    print(df_gold.columns)

    display(df_gold.limit(20))
else:
    print("A tabela Gold não foi encontrada.")

Registros na Gold: 4
Colunas disponíveis na Gold:
['id_treino', 'id_atleta', 'nome_completo', 'peso_kg', 'data_treino', 'modalidade', 'titulo_treino', 'duracao_minutos', 'distancia_km', 'zona_fc_predominante', 'classificacao_intensidade', 'prescricao_carboidrato_intra', 'prescricao_cafeina_pre_treino', 'prescricao_sodio_eletrolitos', 'prescricao_tamponamento_bicarbonato', 'protocolo_cronico_diario', 'total_alertas_clinicos', 'classificacao_risco_clinico', 'tipos_alertas_ativos', 'parecer_farmaceutico_final', 'data_processamento_gold']


id_treino,id_atleta,nome_completo,peso_kg,data_treino,modalidade,titulo_treino,duracao_minutos,distancia_km,zona_fc_predominante,classificacao_intensidade,prescricao_carboidrato_intra,prescricao_cafeina_pre_treino,prescricao_sodio_eletrolitos,prescricao_tamponamento_bicarbonato,protocolo_cronico_diario,total_alertas_clinicos,classificacao_risco_clinico,tipos_alertas_ativos,parecer_farmaceutico_final,data_processamento_gold
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,82.0,2026-09-07,Run,Rodagem + final forte,90.0,14.8,SEM_ZONA,BAIXA,30g/h (piso da faixa 30-60g/h - glicose/maltodextrina),Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade,300 mg/h de sódio (piso conservador) em cápsula ou isotônico,Não indicado para a intensidade da sessão,Creatina: 3g/dia (manutenção) | Beta-Alanina: 3.2g/dia (fracionada) | Ômega-3: 1g/dia EPA+DHA,2,RISCO_CRITICO,SEGURANCA_CARDIOVASCULAR | FARMACO,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos,2026-09-13T19:15:30.422Z
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,82.0,2026-09-08,Run,fartlek,30.0,11.98,SEM_ZONA,BAIXA,Sem necessidade de reposição intra-treino (apenas hidratação),3 mg/kg pré-treino (~246.0 mg) - 45 min antes,Água pura conforme sede habitual,Considerar teste de tolerância: 0.2 g/kg (~16.4 g) fracionado 90 min antes,Creatina: 3g/dia (manutenção) | Beta-Alanina: 3.2g/dia (fracionada) | Ômega-3: 1g/dia EPA+DHA,2,RISCO_CRITICO,SEGURANCA_CARDIOVASCULAR | FARMACO,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos,2026-09-13T19:15:30.422Z
ab11cb42b599b5f2c2b41bbec0d50e2b55099d4d849c4cd82259a40dcd512f6d,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,82.0,2026-09-07,Run,Rodagem + final forte,90.0,14.8,SEM_ZONA,BAIXA,30g/h (piso da faixa 30-60g/h - glicose/maltodextrina),Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade,300 mg/h de sódio (piso conservador) em cápsula ou isotônico,Não indicado para a intensidade da sessão,Creatina: 3g/dia (manutenção) | Beta-Alanina: 3.2g/dia (fracionada) | Ômega-3: 1g/dia EPA+DHA,2,RISCO_CRITICO,SEGURANCA_CARDIOVASCULAR | FARMACO,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos,2026-09-13T19:15:30.422Z
0d6b938e8d495d44ffdf494026896c421162c40899647139869600b555961dda,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,82.0,2026-09-08,Run,fartlek,30.0,11.98,SEM_ZONA,BAIXA,Sem necessidade de reposição intra-treino (apenas hidratação),3 mg/kg pré-treino (~246.0 mg) - 45 min antes,Água pura conforme sede habitual,Considerar teste de tolerância: 0.2 g/kg (~16.4 g) fracionado 90 min antes,Creatina: 3g/dia (manutenção) | Beta-Alanina: 3.2g/dia (fracionada) | Ômega-3: 1g/dia EPA+DHA,2,RISCO_CRITICO,SEGURANCA_CARDIOVASCULAR | FARMACO,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos,2026-09-13T19:15:30.422Z


In [0]:
# Validação da tabela de rastreabilidade dos documentos.
# Esta etapa confirma se os PDFs gerados possuem registros de controle.

nome_fato_documento = "workspace.default.fato_documento_prescricao"

if tabela_existe(nome_fato_documento):
    df_documentos = spark.table(nome_fato_documento)

    print(f"Registros na tabela de documentos: {df_documentos.count()}")
    print("Colunas disponíveis:")
    print(df_documentos.columns)

    display(df_documentos.limit(20))
else:
    print("A tabela de documentos não foi encontrada.")

A tabela de documentos não foi encontrada.


In [0]:
# Verificação dos arquivos PDF na pasta de prescrições pendentes.
# Esta célula apenas lista os arquivos existentes no Volume.

caminho_pdfs_pendentes = (
    "/Volumes/workspace/default/treinos_trainingpeaks/"
    "prescricoes_pendentes"
)

try:
    arquivos_volume = dbutils.fs.ls(caminho_pdfs_pendentes)

    pdfs = [
        arquivo
        for arquivo in arquivos_volume
        if arquivo.name.lower().endswith(".pdf")
    ]

    print(f"Arquivos PDF encontrados: {len(pdfs)}")

    for arquivo in pdfs:
        print(f"- {arquivo.name}")

except Exception as erro:
    print(f"Não foi possível listar a pasta de PDFs: {erro}")

Arquivos PDF encontrados: 5
- RASCUNHO_PLANO_TREINOS_Rafael_Souto_de_Souza_2026-09-07_a_2026-09-08.pdf
- RASCUNHO_Rafael_Souto_de_Souza_20260907_6b56bdd4.pdf
- RASCUNHO_Rafael_Souto_de_Souza_20260907_ab11cb42.pdf
- RASCUNHO_Rafael_Souto_de_Souza_20260908_0d6b938e.pdf
- RASCUNHO_Rafael_Souto_de_Souza_20260908_43eff987.pdf


In [0]:
# Resumo executivo dos principais resultados do MVP.
# Este texto poderá ser utilizado posteriormente na documentação acadêmica.

def valor_indicador(lista_indicadores, nome_indicador):
    # Busca o valor de um indicador pelo nome.
    for item in lista_indicadores:
        if item["indicador"] == nome_indicador:
            return item["valor"]
    return None


print("===== RESUMO EXECUTIVO DO MVP =====")
print(f"Atletas cadastrados: {valor_indicador(indicadores_gerais, 'Atletas na dimensão principal')}")
print(f"Sessões de treinamento: {valor_indicador(indicadores_gerais, 'Sessões de treinamento')}")
print(f"Sessões com zonas: {valor_indicador(indicadores_gerais, 'Sessões com zonas')}")
print(f"Alertas clínicos: {valor_indicador(indicadores_gerais, 'Alertas clínicos')}")
print(f"Prescrições diárias: {valor_indicador(indicadores_gerais, 'Prescrições diárias')}")
print(f"Documentos registrados: {valor_indicador(indicadores_gerais, 'Documentos de prescrição')}")
print()
print("A análise foi executada em modo somente leitura.")
print("Os resultados devem ser revisados antes de serem transcritos para o README.")

===== RESUMO EXECUTIVO DO MVP =====
Atletas cadastrados: 3
Sessões de treinamento: 4
Sessões com zonas: 2
Alertas clínicos: 8
Prescrições diárias: 4
Documentos registrados: None

A análise foi executada em modo somente leitura.
Os resultados devem ser revisados antes de serem transcritos para o README.


In [0]:
# Encerramento da análise final.
# Esta célula registra que o notebook de validação foi executado.

print("Análise final do MVP concluída.")
print("As tabelas foram consultadas sem alteração dos dados.")
print("As evidências visuais desta execução podem ser utilizadas na documentação.")

Análise final do MVP concluída.
As tabelas foram consultadas sem alteração dos dados.
As evidências visuais desta execução podem ser utilizadas na documentação.
